In [ ]:
import os
import json
import torch
import numpy as np
import PIL
from PIL import Image
from IPython.display import HTML
from pipeline import PyramidDiTForVideoGeneration

In [ ]:
# variant='diffusion_transformer_384p'       # For low resolution variant
variant='diffusion_transformer_768p'     # For high resolution variant

model_name = "pyramid_flux"   # select the model "pyramid_flux" or "pyramid_mmdit"

# model_path = "/home/jinyang06/models/pyramid-flow-miniflux"   # The downloaded checkpoint dir
model_path = "/path/to/your/finetuned/output_folder"
model_dtype = 'bf16'

device_id = 0
torch.cuda.set_device(device_id)

model = PyramidDiTForVideoGeneration(
    model_path,
    model_dtype,
    model_name=model_name,
    model_variant=variant,
)



In [ ]:
model.vae.to("cuda")
model.dit.to("cuda")
model.text_encoder.to("cuda")

model.vae.enable_tiling()

if model_dtype == "bf16":
    torch_dtype = torch.bfloat16 
elif model_dtype == "fp16":
    torch_dtype = torch.float16
else:
    torch_dtype = torch.float32


def resize_crop_image(img: PIL.Image.Image, tgt_width, tgt_height):
    ori_width, ori_height = img.width, img.height
    scale = max(tgt_width / ori_width, tgt_height / ori_height)
    resized_width = round(ori_width * scale)
    resized_height = round(ori_height * scale)
    img = img.resize((resized_width, resized_height), resample=PIL.Image.LANCZOS)

    left = (resized_width - tgt_width) / 2
    top = (resized_height - tgt_height) / 2
    right = (resized_width + tgt_width) / 2
    bottom = (resized_height + tgt_height) / 2

    # Crop the center of the image
    img = img.crop((left, top, right, bottom))
    
    return img